# Weekly Update Generator - AgentCore Deployment

## Overview

In this tutorial we will learn how to build and deploy an automated weekly status report generator using Amazon Bedrock AgentCore Runtime. The agent collects data from multiple sources (team updates, meeting notes, metrics, bug trackers), performs analysis, generates visualizations, and uploads comprehensive reports to S3.

### Architecture & Files

![Architecture Diagram](./images/architecture.png)

- **Agent Implementation**: `01_weekly_report_generator_async/agent/agent.py` - Main agent file that defines the Bedrock Agent Core application, agent configuration, and entrypoint
- **Tools**: `01_weekly_report_generator_async/agent/tools.py` - Contains 16 tools for data reading, analysis, visualization, and S3 upload
- **Demo Data**: `01_weekly_report_generator_async/demo_data/` directory containing:
  - `project_status/` - CSV files with project progress and blockers
  - `team_updates/` - Markdown files with individual team member updates
  - `metrics/` - CSV files with KPI data (current and historical)
  - `issues/` - JSON files with bug tracker data
  - `meeting_notes/` - Markdown files with meeting summaries

### How It Works

1. **Data Collection** - The agent dynamically discovers and reads the latest week's data:
   - Scans directories for files matching patterns like `projects_week_XX.csv`, `kpis_week_XX.csv`, `bug_tracker_week_XX.json`
   - Automatically uses the most recent week number found
   - Reads all team member markdown files and meeting notes

2. **Data Analysis** - The agent performs intelligent analysis:
   - Validates data quality and completeness across all sources
   - Cross-references information (e.g., matching project names in status files vs team updates)
   - Performs sentiment analysis on team updates to detect morale issues
   - Calculates risk scores based on project health, blockers, and bug severity

3. **Visualization Generation** - Creates PNG charts using matplotlib:
   - Bug severity distribution (pie/bar charts)
   - KPI metrics trends with historical comparison
   - Project timeline and progress visualization
   - Team velocity charts
   - Predictive forecast models for key metrics

4. **Report Synthesis** - Compiles a structured markdown report with:
   - Executive summary with key highlights and concerns
   - Detailed sections for projects, team updates, KPIs, bugs, and meetings
   - Risk analysis and blockers
   - Action items and next week priorities

5. **S3 Upload** - Automatically uploads to S3:
   - Final markdown report
   - All generated chart images
   - Organized by date in the S3 bucket for easy retrieval

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Asynchronous agent                                                       |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 4.5                                                        |
| Tutorial components | Multi-tool agent, data analysis, visualization, S3 integration, AgentCore Runtime|
| Tutorial vertical   | Business Operations & Reporting                                                  |
| Example complexity  | Intermediate                                                                     |
| SDK used            | Amazon BedrockAgentCore Python SDK, boto3, matplotlib, scikit-learn              |

### Tutorial Architecture

This tutorial demonstrates how to deploy an asynchronous reporting agent to AgentCore runtime. The agent orchestrates 16 different tools to create comprehensive weekly status reports automatically.

### Tutorial Key Features

* Hosting an asychronous, multi-tool agent on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models (Claude Sonnet 4)
* Using Strands Agents SDK
* S3 integration for data storage and report delivery

### Deployment

The agent runs as a Bedrock Agent Core application that can be:
- Invoked locally for testing
- Deployed to AWS Lambda for production use
- Called via API with async task tracking (returns immediately while processing in background)
- Monitored via ping endpoint that reports HEALTHY or HEALTHY_BUSY status


## Prerequisites
- AWS Account with access to Amazon Bedrock AgentCore
- AWS credentials
- Python 3.12+
- S3 bucket for storing demo data and reports

## Setup and Imports

In [ ]:
from boto3.session import Session
import time
import json
import boto3
import re
from datetime import datetime, timedelta
from pathlib import Path
from botocore.exceptions import ClientError

print("✅ Imports successful!")

## Pre-deployment Configuration

In [ ]:
boto_session = Session()
region = boto_session.region_name
agent_name = "weekly_update_generator"

# TODO: Replace with your S3 bucket name
S3_BUCKET = "YOUR-BUCKET-NAME"
S3_PREFIX = "demo_data"

print(f"📍 Region: {region}")
print(f"🤖 Agent name: {agent_name}")
print(f"🪣 S3 Bucket: {S3_BUCKET}")

## Step 1: Update Demo Data and Upload to S3

In [ ]:
# Run the update script
!python update_demo_dates.py --bucket {S3_BUCKET} --prefix {S3_PREFIX}

## Step 2: Deploy Agent to AgentCore

The CreateAgentRuntime operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent.

In this tutorial can will the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Async Agent Implementation

The `01_weekly_report_generator_async/agent/agent.py` file implements an asynchronous agent that handles long-running report generation tasks. When invoked, the agent immediately returns a task ID and processes the request in a background thread. This allows the caller to continue without waiting for the entire report generation to complete.

The agent tracks active tasks using a counter and implements a `@app.ping` handler that reports the agent's status:

### Understanding the /ping Endpoint

The `/ping` endpoint is crucial for monitoring async agents. It returns one of two statuses:

- **`{"status": "Healthy"}`** - Agent is idle and ready to accept new tasks
- **`{"status": "HealthyBusy"}`** - Agent is actively processing tasks but still responsive

This allows clients to:
- Poll the agent to determine when tasks complete
- Monitor agent health without interfering with ongoing work
- Implement retry logic based on agent availability

For more details, see the [Asynchronous Agents documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-long-run.html).

### Configure AgentCore Runtime deployment
First we will use our AgentCore SDK to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the AgentCore SDK to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. Tools are packaged with the agent so that the agent can access them at runtime.

![](images/configure.png)

In [ ]:
# Change to agent directory for configuration
import os
os.chdir('agent')
print(f"Working directory: {os.getcwd()}")

### Configure the agent

In [ ]:
import boto3
import json
import zipfile
import tempfile
import os
import time

boto_session = boto3.session.Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=region)

def create_or_get_execution_role(agent_name, region, account_id):
    iam = boto3.client('iam')
    role_name = f"AgentCoreRuntime-{agent_name[:40]}"
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {"StringEquals": {"aws:SourceAccount": account_id},
                          "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/*"}}}]
    }
    perms = {"Version": "2012-10-17", "Statement": [
        {"Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogDelivery", "logs:PutLogEvents",
                                       "logs:CreateLogStream"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["s3:PutObject", "s3:GetObject", "s3:ListBucket"], "Resource": "*"}
    ]}
    try:
        role = iam.create_role(RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust_policy))
        iam.put_role_policy(RoleName=role_name, PolicyName="AgentCoreRuntimePermissions",
                            PolicyDocument=json.dumps(perms))
        print(f"Created IAM role: {role_name}")
        time.sleep(10)
        return role['Role']['Arn']
    except iam.exceptions.EntityAlreadyExistsException:
        return iam.get_role(RoleName=role_name)['Role']['Arn']

def package_and_upload_to_s3(agent_name, files, region, account_id):
    s3 = boto3.client('s3', region_name=region)
    bucket_name = f"bedrock-agentcore-{account_id}-{region}"
    try:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})
    except s3.exceptions.BucketAlreadyOwnedByYou:
        pass
    with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmpf:
        zip_path = tmpf.name
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            if os.path.exists(f):
                zf.write(f, os.path.basename(f))
    s3_key = f"{agent_name}/deployment.zip"
    s3.upload_file(zip_path, bucket_name, s3_key)
    os.unlink(zip_path)
    return bucket_name, s3_key

role_arn = create_or_get_execution_role(agent_name, region, account_id)
bucket_name, s3_key = package_and_upload_to_s3(agent_name, ["agent.py", "requirements.txt"], region, account_id)
print(f"Agent configured: {agent_name}")
print(f"Role ARN: {role_arn}")


### Launch the agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

![](images/launch.png)

In [ ]:
# Create the AgentCore Runtime using the CodeZip deployment type
create_response = agentcore_control.create_agent_runtime(
    agentRuntimeName=agent_name,
    agentRuntimeArtifact={
        'codeConfiguration': {
            'code': {'s3': {'bucket': bucket_name, 'prefix': s3_key}},
            'runtime': 'PYTHON_3_11',
            'entryPoint': ['agent.py']
        }
    },
    roleArn=role_arn,
    networkConfiguration={'networkMode': 'PUBLIC'}
)
agent_runtime_id = create_response['agentRuntimeId']
agent_runtime_arn = create_response['agentRuntimeArn']
print(f"AgentCore Runtime created:")
print(f"  Runtime ID:  {agent_runtime_id}")
print(f"  Runtime ARN: {agent_runtime_arn}")


## Step 3: Wait for Agent to be Ready

Let's check the agent's deployment status to ensure it can be invoked succesfully

In [ ]:
import time
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
status = create_response.get('status', 'CREATING')
while status not in end_status:
    time.sleep(15)
    r = agentcore_control.get_agent_runtime(agentRuntimeId=agent_runtime_id)
    status = r['status']
    print(f"Status: {status}")
print(f"\nFinal status: {status}")


## Step 4: Add S3 Permissions to Execution Role

For this tutorial, we will read and write data stored in an Amazon S3 bucket. Now that our agent has been launched, we need to give the agent permission to access the S3 bucket.

In [ ]:
# Get execution role from agent runtime
agent_runtime_id = agent_runtime_id
print(f"Agent Runtime ID: {agent_runtime_id}")

agentcore_client = boto3.client('bedrock-agentcore-control', region_name=region)


response = agentcore_client.get_agent_runtime(
    agentRuntimeId=agent_runtime_id,
    agentRuntimeVersion='1'
)

execution_role_arn = response.get('roleArn')

execution_role_name = execution_role_arn.split('/')[-1]
print(f"✓ Execution role: {execution_role_name}")

iam_client = boto3.client('iam')
policy_document = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
        "Resource": [f"arn:aws:s3:::{S3_BUCKET}", f"arn:aws:s3:::{S3_BUCKET}/*"]
    }]
}

iam_client.put_role_policy(
    RoleName=execution_role_name,
    PolicyName='WeeklyReportsS3Access',
    PolicyDocument=json.dumps(policy_document)
)

print(f"\n✅ S3 permissions added!")
print(f"   Role: {execution_role_name}")
print(f"   Bucket: {S3_BUCKET}")
    


## Step 5: Invoke the Agent

Finally, we can invoke our agent and have it generate our weekly report

In [ ]:
import boto3
import json
from IPython.display import Markdown, display

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)

invoke_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How is the weather now in Athens ?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = invoke_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

try:
    events = []
    for event in invoke_response.get("response", []):
        events.append(event)
except Exception as e:
    events = [f"Error reading EventStream: {e}"]
response_text = json.loads(events[0].decode("utf-8"))
display(Markdown(response_text))


## 6. View the Agent's Output

Once the agent has created the weekly report, you will find it in S3 under the path of s3:/{your-bucket-name}/weekly_reports/{year}/{week}/weekly_report.md.

![](images/report.png)

## Cleanup (Optional)
Let's now clean up the AgentCore Runtime created

In [ ]:
print(f"Agent Runtime ID:  {agent_runtime_id}")
print(f"Agent Runtime ARN: {agent_runtime_arn}")


In [ ]:
print(f"Agent Runtime ID:  {agent_runtime_id}")
print(f"Agent Runtime ARN: {agent_runtime_arn}")
